# Decoder型言語モデルによる文章生成

このNotebookでは、Ministralのdecoder型言語モデルを読み込み、ユーザーの質問に対する回答を生成する。

最初のコードセルでは、次の準備を行う。

- `AutoProcessor`：チャット形式の文章をモデルが扱えるtoken IDへ変換し、生成後のtoken IDを文章へ戻す。
- `Mistral3ForConditionalGeneration`：入力されたtoken列に続くtokenを逐次予測する。
- `torch_dtype=torch.bfloat16`：モデルの重みを低精度で保持し、メモリ使用量を削減する。
- `device_map="auto"`：利用可能なCPUやGPUへモデルを自動配置する。
- `messages`：モデルへ渡すユーザー質問をチャット形式で定義する。

In [2]:
import torch
from transformers import AutoProcessor, Mistral3ForConditionalGeneration

model_id = "mistralai/Ministral-3-3B-Instruct-2512"

processor = AutoProcessor.from_pretrained(model_id)

model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": "Explain Tokyo"}],
    }
]

processor_config.json:   0%|          | 0.00/976 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/21.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

[transformers] The tokenizer you are loading from 'mistralai/Ministral-3-3B-Instruct-2512' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


model.safetensors:   0%|          | 0.00/4.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

## Tokenizationと文章生成

`apply_chat_template`は、`messages`をモデル固有のチャット形式へ整形し、token ID列に変換する。  
`reasoning_effort="none"`により、明示的な推論過程を生成しない設定にしている。

`generate`では、入力token列 $x$ に続く出力token $y_t$ を、直前までの出力に条件付けて逐次生成する。  
各時点の予測は $p(y_t \mid x, y_1, \ldots, y_{t-1})$ に基づき、`max_new_tokens=200`は新しく生成するtoken数の上限を200に設定する。

`outputs`には入力tokenと生成tokenの両方が含まれる。  
そのため、`inputs["input_ids"].shape[-1]`までを除いて生成部分だけを取り出し、`decode`で文章へ戻して表示する。

In [3]:
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="none",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
)

print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `reasoning_effort` is not a valid argument for this processor and will be ignored.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Tokyo is the capital and largest city of Japan, known for its **cutting-edge technology, vibrant culture, and unique blend of tradition and modernity**. Here’s a detailed breakdown of what makes Tokyo so fascinating:

---

### **1. Geography & Location**
- **Size & Density**: Tokyo spans **2,194 km²** (about the size of Singapore) but has a population of **14+ million** in the city proper and **37+ million** in the Greater Tokyo Area (one of the most densely populated regions in the world).
- **Climate**: Four distinct seasons—**hot, humid summers (June–August)**, **cool winters (December–February)**, **rainy season (May–June)**, and **spring (March–April)**.
- **Landmarks**:
  - **Mount Fuji** (Japan’s highest peak, ~1,200 km away) is visible on clear days.
  - **Shinjuku Gyoen** (a stunning park with Japanese, French, and English gardens).
  - **Ueno Park** (home to museums, temples, and cherry blossoms in spring).

---

### **2. Culture & Traditions**
Tokyo is a **melting pot of anc